In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import zipfile
import re
import json

In [ ]:
zip_path = "/content/amlh data.zip"   # change if your zip name is slightly different
extract_path = "/content/flu_data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Files extracted successfully!")

Files extracted successfully!


In [ ]:
all_files = glob.glob(os.path.join(extract_path, "*.csv"))
print("Total CSV files found:", len(all_files))

for f in all_files[:10]:
    print(os.path.basename(f))

Total CSV files found: 30
region6_2019_20_StackedColumnChart_Data.csv
region4_2023_24_LineChart_Data.csv
region4_2023_24_StackedColumnChart_Data.csv
region4_2022_23_StackedColumnChart_Data.csv
region9_2021_22_LineChart_Data.csv
region9_2022_23_StackedColumnChart_Data.csv
region4_2019_20_LineChart_Data.csv
region4_2024_25_StackedColumnChart_Data.csv
region6_2021_22_LineChart_Data.csv
region4_2022_23_LineChart_Data.csv


In [ ]:
linechart_files = [f for f in all_files if "LineChart" in os.path.basename(f)]
stacked_files = [f for f in all_files if "StackedColumnChart" in os.path.basename(f)]

print("LineChart files:", len(linechart_files))
print("StackedColumnChart files:", len(stacked_files))

LineChart files: 15
StackedColumnChart files: 15


In [ ]:
sample_file = linechart_files[0]
print("Sample file:", os.path.basename(sample_file))

sample_df = pd.read_csv(sample_file, skiprows=1)
sample_df.head()

Sample file: region4_2023_24_LineChart_Data.csv


,YEAR,WEEK,AGE 0-4,AGE 5-24,AGE 25-49,AGE 25-64,AGE 50-64,AGE 65,ILITOTAL,TOTAL PATIENTS,NUM. OF PROVIDERS,%UNWEIGHTED ILI,% WEIGHTED ILI
0,2023,40,5266,6572,3487,NaN,1302,1183,17810,593391,952,3.00139,2.86023
1,2023,41,5613,6538,3683,NaN,1341,1199,18374,586813,960,3.13115,2.98154
2,2023,42,5845,7079,3983,NaN,1269,1192,19368,580753,947,3.33498,3.15020
3,2023,43,6271,8294,4454,NaN,1531,1426,21976,601701,955,3.65231,3.46040
4,2023,44,6715,9232,4826,NaN,1513,1351,23637,594347,953,3.97697,3.79498


In [ ]:
skiprows=1

In [ ]:
sample_df.columns

Index(['YEAR', 'WEEK', 'AGE 0-4', 'AGE 5-24', 'AGE 25-49', 'AGE 25-64',
       'AGE 50-64', 'AGE 65', 'ILITOTAL', 'TOTAL PATIENTS',
       'NUM. OF PROVIDERS', '%UNWEIGHTED ILI', '% WEIGHTED ILI'],
      dtype='object')

In [ ]:
def extract_region_season(filename):
    name = os.path.basename(filename)

    # Extract region number
    region_match = re.search(r"region(\d+)", name)
    region = f"Region {region_match.group(1)}" if region_match else None

    # Extract season
    season_match = re.search(r"(\d{4}_\d{2})", name)
    season = season_match.group(1) if season_match else None

    return region, season

In [ ]:
for f in linechart_files[:5]:
    print(os.path.basename(f), "→", extract_region_season(f))

region4_2023_24_LineChart_Data.csv → ('Region 4', '2023_24')
region9_2021_22_LineChart_Data.csv → ('Region 9', '2021_22')
region4_2019_20_LineChart_Data.csv → ('Region 4', '2019_20')
region6_2021_22_LineChart_Data.csv → ('Region 6', '2021_22')
region4_2022_23_LineChart_Data.csv → ('Region 4', '2022_23')


In [ ]:
ili_dfs = []

for file in linechart_files:
    df = pd.read_csv(file, skiprows=1)

    region, season = extract_region_season(file)

    df["Region"] = region
    df["Season"] = season

    ili_dfs.append(df)

ili_data = pd.concat(ili_dfs, ignore_index=True)

print("Combined ILINet shape:", ili_data.shape)
ili_data.head()

Combined ILINet shape: (780, 15)


,YEAR,WEEK,AGE 0-4,AGE 5-24,AGE 25-49,AGE 25-64,AGE 50-64,AGE 65,ILITOTAL,TOTAL PATIENTS,NUM. OF PROVIDERS,%UNWEIGHTED ILI,% WEIGHTED ILI,Region,Season
0,2023,40,5266,6572,3487,NaN,1302,1183,17810,593391,952,3.00139,2.86023,Region 4,2023_24
1,2023,41,5613,6538,3683,NaN,1341,1199,18374,586813,960,3.13115,2.98154,Region 4,2023_24
2,2023,42,5845,7079,3983,NaN,1269,1192,19368,580753,947,3.33498,3.15020,Region 4,2023_24
3,2023,43,6271,8294,4454,NaN,1531,1426,21976,601701,955,3.65231,3.46040,Region 4,2023_24
4,2023,44,6715,9232,4826,NaN,1513,1351,23637,594347,953,3.97697,3.79498,Region 4,2023_24


In [ ]:
ili_data.isnull().sum()

,0
YEAR,0
WEEK,0
AGE 0-4,0
AGE 5-24,0
AGE 25-49,0
AGE 25-64,780
AGE 50-64,0
AGE 65,0
ILITOTAL,0
TOTAL PATIENTS,0


In [ ]:
ili_clean = ili_data[[
    "YEAR",
    "WEEK",
    "ILITOTAL",
    "TOTAL PATIENTS",
    "% WEIGHTED ILI",
    "Region",
    "Season"
]].copy()

ili_clean.head()

,YEAR,WEEK,ILITOTAL,TOTAL PATIENTS,% WEIGHTED ILI,Region,Season
0,2023,40,17810,593391,2.86023,Region 4,2023_24
1,2023,41,18374,586813,2.98154,Region 4,2023_24
2,2023,42,19368,580753,3.15020,Region 4,2023_24
3,2023,43,21976,601701,3.46040,Region 4,2023_24
4,2023,44,23637,594347,3.79498,Region 4,2023_24


In [ ]:
ili_clean.rename(columns={
    "YEAR": "Year",
    "WEEK": "Week",
    "ILITOTAL": "ILI_Total",
    "TOTAL PATIENTS": "Total_Patients",
    "% WEIGHTED ILI": "Weighted_ILI"
}, inplace=True)

ili_clean.head()

,Year,Week,ILI_Total,Total_Patients,Weighted_ILI,Region,Season
0,2023,40,17810,593391,2.86023,Region 4,2023_24
1,2023,41,18374,586813,2.98154,Region 4,2023_24
2,2023,42,19368,580753,3.15020,Region 4,2023_24
3,2023,43,21976,601701,3.46040,Region 4,2023_24
4,2023,44,23637,594347,3.79498,Region 4,2023_24


In [ ]:
ili_clean["Week"] = ili_clean["Week"].astype(int)
ili_clean["Year"] = ili_clean["Year"].astype(int)

ili_clean["Date"] = pd.to_datetime(
    ili_clean["Year"].astype(str) + "-W" + ili_clean["Week"].astype(str) + "-1",
    format="%G-W%V-%u",
    errors="coerce"
)

ili_clean.head()

,Year,Week,ILI_Total,Total_Patients,Weighted_ILI,Region,Season,Date
0,2023,40,17810,593391,2.86023,Region 4,2023_24,2023-10-02
1,2023,41,18374,586813,2.98154,Region 4,2023_24,2023-10-09
2,2023,42,19368,580753,3.15020,Region 4,2023_24,2023-10-16
3,2023,43,21976,601701,3.46040,Region 4,2023_24,2023-10-23
4,2023,44,23637,594347,3.79498,Region 4,2023_24,2023-10-30


In [ ]:
ili_clean = ili_clean.sort_values(by=["Region", "Date"]).reset_index(drop=True)
ili_clean.head()

,Year,Week,ILI_Total,Total_Patients,Weighted_ILI,Region,Season,Date
0,2019,40,5637,333469,1.67601,Region 4,2019_20,2019-09-30
1,2019,41,5349,317260,1.63170,Region 4,2019_20,2019-10-07
2,2019,42,5711,311644,1.81532,Region 4,2019_20,2019-10-14
3,2019,43,6683,322052,1.96609,Region 4,2019_20,2019-10-21
4,2019,44,7119,318970,2.29988,Region 4,2019_20,2019-10-28


In [ ]:
print("Final ILINet cleaned shape:", ili_clean.shape)
ili_clean.info()

Final ILINet cleaned shape: (780, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 780 entries, 0 to 779
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Year            780 non-null    int64         
 1   Week            780 non-null    int64         
 2   ILI_Total       780 non-null    int64         
 3   Total_Patients  780 non-null    int64         
 4   Weighted_ILI    780 non-null    float64       
 5   Region          780 non-null    object        
 6   Season          780 non-null    object        
 7   Date            780 non-null    datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(4), object(2)
memory usage: 48.9+ KB


In [ ]:
ili_clean.to_csv("/content/clean_ilinet_data.csv", index=False)
print("Saved as clean_ilinet_data.csv")

Saved as clean_ilinet_data.csv


In [ ]:
pop_raw = pd.read_excel("/content/state_population_2024.xlsx", header=None)
pop_raw.head(12)

,0,1,2,3,4,5,6
0,table with row headers in column A and column ...,NaN,NaN,NaN,NaN,NaN,NaN
1,Annual Estimates of the Resident Population fo...,NaN,NaN,NaN,NaN,NaN,NaN
2,Geographic Area,"April 1, 2020 Estimates Base",Population Estimate (as of July 1),NaN,NaN,NaN,NaN
3,NaN,NaN,2020,2021.0,2022.0,2023.0,2024.0
4,United States,331515736,331577720,332099760.0,334017321.0,336806231.0,340110988.0
5,Northeast,57617706,57431458,57252533.0,57159597.0,57398303.0,57832935.0
6,Midwest,68998970,68984258,68872831.0,68903297.0,69186401.0,69596584.0
7,South,126281537,126476549,127368010.0,129037849.0,130893358.0,132665693.0
8,West,78617523,78685455,78606386.0,78916578.0,79328169.0,80015776.0
9,.Alabama,5025369,5033094,5049196.0,5076181.0,5117673.0,5157699.0


In [ ]:
pop = pd.read_excel("/content/state_population_2024.xlsx", skiprows=3)
pop.head(15)

,Unnamed: 0,Unnamed: 1,2020,2021,2022,2023,2024
0,United States,331515736.0,331577720.0,332099760.0,334017321.0,336806231.0,340110988.0
1,Northeast,57617706.0,57431458.0,57252533.0,57159597.0,57398303.0,57832935.0
2,Midwest,68998970.0,68984258.0,68872831.0,68903297.0,69186401.0,69596584.0
3,South,126281537.0,126476549.0,127368010.0,129037849.0,130893358.0,132665693.0
4,West,78617523.0,78685455.0,78606386.0,78916578.0,79328169.0,80015776.0
5,.Alabama,5025369.0,5033094.0,5049196.0,5076181.0,5117673.0,5157699.0
6,.Alaska,733395.0,733017.0,734420.0,734442.0,736510.0,740133.0
7,.Arizona,7158110.0,7187135.0,7274078.0,7377566.0,7473027.0,7582384.0
8,.Arkansas,3011553.0,3014546.0,3026870.0,3047704.0,3069463.0,3088354.0
9,.California,39555674.0,39521958.0,39142565.0,39142414.0,39198693.0,39431263.0


In [ ]:
pop.columns

Index(['Unnamed: 0', 'Unnamed: 1', 2020, 2021, 2022, 2023, 2024], dtype='object')

In [ ]:
pop_clean = pop.iloc[:, [0, 6]].copy()
pop_clean.columns = ["State", "Population_2024"]

pop_clean.head(20)

,State,Population_2024
0,United States,340110988.0
1,Northeast,57832935.0
2,Midwest,69596584.0
3,South,132665693.0
4,West,80015776.0
5,.Alabama,5157699.0
6,.Alaska,740133.0
7,.Arizona,7582384.0
8,.Arkansas,3088354.0
9,.California,39431263.0


In [ ]:
# Keep rows that start with "."
pop_clean = pop_clean[pop_clean["State"].astype(str).str.startswith(".")].copy()

# Remove the dot
pop_clean["State"] = pop_clean["State"].str.replace(".", "", regex=False).str.strip()

pop_clean.head(20)

,State,Population_2024
5,Alabama,5157699.0
6,Alaska,740133.0
7,Arizona,7582384.0
8,Arkansas,3088354.0
9,California,39431263.0
10,Colorado,5957493.0
11,Connecticut,3675069.0
12,Delaware,1051917.0
13,District of Columbia,702250.0
14,Florida,23372215.0


In [ ]:
print("Number of states found:", len(pop_clean))
pop_clean.head()

Number of states found: 52


,State,Population_2024
5,Alabama,5157699.0
6,Alaska,740133.0
7,Arizona,7582384.0
8,Arkansas,3088354.0
9,California,39431263.0


In [ ]:
region_map = {
    "Alabama": "Region 4",
    "Florida": "Region 4",
    "Georgia": "Region 4",
    "Kentucky": "Region 4",
    "Mississippi": "Region 4",
    "North Carolina": "Region 4",
    "South Carolina": "Region 4",
    "Tennessee": "Region 4",

    "Arkansas": "Region 6",
    "Louisiana": "Region 6",
    "New Mexico": "Region 6",
    "Oklahoma": "Region 6",
    "Texas": "Region 6",

    "Arizona": "Region 9",
    "California": "Region 9",
    "Hawaii": "Region 9",
    "Nevada": "Region 9"
}

In [ ]:
pop_clean["Region"] = pop_clean["State"].map(region_map)

region_pop_states = pop_clean[pop_clean["Region"].notna()].copy()
region_pop_states.head(20)

,State,Population_2024,Region
5,Alabama,5157699.0,Region 4
7,Arizona,7582384.0,Region 9
8,Arkansas,3088354.0,Region 6
9,California,39431263.0,Region 9
14,Florida,23372215.0,Region 4
15,Georgia,11180878.0,Region 4
16,Hawaii,1446146.0,Region 9
22,Kentucky,4588372.0,Region 4
23,Louisiana,4597740.0,Region 6
29,Mississippi,2943045.0,Region 4


In [ ]:
region_population = region_pop_states.groupby("Region", as_index=False)["Population_2024"].sum()
region_population

,Region,Population_2024
0,Region 4,70994814.0
1,Region 6,45202574.0
2,Region 9,51727260.0


In [ ]:
region_population.to_csv("/content/region_population_2024.csv", index=False)
print("Saved as region_population_2024.csv")

Saved as region_population_2024.csv


In [ ]:
master_data = ili_clean.merge(region_population, on="Region", how="left")
master_data.head()

,Year,Week,ILI_Total,Total_Patients,Weighted_ILI,Region,Season,Date,Population_2024
775,2025,35,8643,409159,2.07792,Region 9,2024_25,2025-08-25,51727260.0
776,2025,36,8286,371227,2.21147,Region 9,2024_25,2025-09-01,51727260.0
777,2025,37,8530,408760,2.08559,Region 9,2024_25,2025-09-08,51727260.0
778,2025,38,8226,409936,2.00848,Region 9,2024_25,2025-09-15,51727260.0
779,2025,39,8193,409146,2.02152,Region 9,2024_25,2025-09-22,51727260.0


In [ ]:
master_data["ILI_per_100k"] = (master_data["ILI_Total"] / master_data["Population_2024"]) * 100000
master_data.head()

,Year,Week,ILI_Total,Total_Patients,Weighted_ILI,Region,Season,Date,Population_2024,ILI_per_100k
0,2019,40,5637,333469,1.67601,Region 4,2019_20,2019-09-30,70994814.0,7.940017
1,2019,41,5349,317260,1.63170,Region 4,2019_20,2019-10-07,70994814.0,7.534353
2,2019,42,5711,311644,1.81532,Region 4,2019_20,2019-10-14,70994814.0,8.044250
3,2019,43,6683,322052,1.96609,Region 4,2019_20,2019-10-21,70994814.0,9.413364
4,2019,44,7119,318970,2.29988,Region 4,2019_20,2019-10-28,70994814.0,10.027493


In [ ]:
print("Master dataset shape:", master_data.shape)
master_data.info()
master_data.head()

Master dataset shape: (780, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 780 entries, 0 to 779
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Year             780 non-null    int64         
 1   Week             780 non-null    int64         
 2   ILI_Total        780 non-null    int64         
 3   Total_Patients   780 non-null    int64         
 4   Weighted_ILI     780 non-null    float64       
 5   Region           780 non-null    object        
 6   Season           780 non-null    object        
 7   Date             780 non-null    datetime64[ns]
 8   Population_2024  780 non-null    float64       
 9   ILI_per_100k     780 non-null    float64       
dtypes: datetime64[ns](1), float64(3), int64(4), object(2)
memory usage: 61.1+ KB


,Year,Week,ILI_Total,Total_Patients,Weighted_ILI,Region,Season,Date,Population_2024,ILI_per_100k
0,2019,40,5637,333469,1.67601,Region 4,2019_20,2019-09-30,70994814.0,7.940017
1,2019,41,5349,317260,1.63170,Region 4,2019_20,2019-10-07,70994814.0,7.534353
2,2019,42,5711,311644,1.81532,Region 4,2019_20,2019-10-14,70994814.0,8.044250
3,2019,43,6683,322052,1.96609,Region 4,2019_20,2019-10-21,70994814.0,9.413364
4,2019,44,7119,318970,2.29988,Region 4,2019_20,2019-10-28,70994814.0,10.027493


In [ ]:
master_data.to_csv("/content/master_influenza_dataset.csv", index=False)
print("Saved as master_influenza_dataset.csv")

Saved as master_influenza_dataset.csv
